# 06-P5. paper-aligned agent contract

이 노트북은 paper-aligned anomaly output을 Agent / Priority Engine 계약 스키마로 변환한다.

현재 목적:
- anomaly score와 criticality 결과를 운영 판단 점수로 정규화
- 관련 fault / disturbance history를 같은 row에 붙임
- Agent가 바로 읽을 수 있는 `main_abnormal_features`, `feature_explanation`, `priority_reason`를 생성


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다.')


def risk_level_from_score(score: float) -> str:
    if score >= 0.75:
        return 'critical'
    if score >= 0.50:
        return 'high'
    if score >= 0.30:
        return 'medium'
    return 'low'


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned'
ALIGNMENT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'label_alignment'

SUMMARY_PATH = OUTPUT_DIR / 'event_detection_summary.csv'
SCORES_PATH = OUTPUT_DIR / 'autoencoder_reconstruction_scores.csv'
MAIN_FEATURES_PATH = OUTPUT_DIR / 'main_abnormal_features.csv'
FAULT_ALIGNMENT_PATH = ALIGNMENT_DIR / 'fault_alignment.csv'
DISTURBANCE_ALIGNMENT_PATH = ALIGNMENT_DIR / 'disturbance_alignment.csv'

AGENT_OUTPUT = OUTPUT_DIR / 'agent_contract_output.csv'
PRIORITY_OUTPUT = OUTPUT_DIR / 'priority_engine_input.csv'
SCHEMA_OUTPUT = OUTPUT_DIR / 'agent_contract_schema.json'
METADATA_OUTPUT = OUTPUT_DIR / 'agent_contract_metadata.json'

summary_df = pd.read_csv(SUMMARY_PATH, parse_dates=['report_date', 'event_start', 'event_end', 'detection_time'])
scores_df = pd.read_csv(SCORES_PATH, parse_dates=['window_start', 'window_end', 'report_date', 'event_start', 'event_end'])
main_features_df = pd.read_csv(MAIN_FEATURES_PATH)
fault_alignment = pd.read_csv(FAULT_ALIGNMENT_PATH, parse_dates=['report_date', 'window_start', 'window_end'])
disturbance_alignment = pd.read_csv(DISTURBANCE_ALIGNMENT_PATH, parse_dates=['event_start', 'window_start', 'window_end'])

fault_alignment = fault_alignment[fault_alignment['is_usable'] == True].copy()
disturbance_alignment = disturbance_alignment[disturbance_alignment['is_usable'] == True].copy()

event_score_agg = (
    scores_df[scores_df['selected_for_event_eval'] == True]
    .groupby(['manufacturer', 'event_type', 'event_split', 'event_id'], dropna=False)
    .agg(
        max_anomaly_score=('anomaly_score', 'max'),
        mean_anomaly_score=('anomaly_score', 'mean'),
        p99_above_count=('is_above_train_p099', 'sum'),
        selected_window_count=('anomaly_score', 'size'),
    )
    .reset_index()
)

contract_df = summary_df.merge(
    event_score_agg,
    on=['manufacturer', 'event_type', 'event_split', 'event_id'],
    how='left',
    validate='one_to_one',
)
main_feature_merge_columns = [
    'manufacturer',
    'event_type',
    'event_split',
    'event_id',
    'substation_id',
    'configuration_type',
    'fault_label',
    'focus_rule',
    'focus_window_count',
    'main_abnormal_features',
    'feature_explanation',
    'top_feature_1',
    'top_feature_1_direction',
    'top_feature_1_contribution',
    'top_feature_2',
    'top_feature_2_direction',
    'top_feature_2_contribution',
    'top_feature_3',
    'top_feature_3_direction',
    'top_feature_3_contribution',
]
contract_df = contract_df.merge(
    main_features_df[main_feature_merge_columns],
    on=['manufacturer', 'event_type', 'event_split', 'event_id', 'substation_id', 'configuration_type', 'fault_label'],
    how='left',
    validate='one_to_one',
)

reference_timestamp = contract_df['detection_time'].where(contract_df['detected'], contract_df['event_end'])
contract_df['timestamp'] = reference_timestamp

related_fault_history = []
related_disturbance_history = []
recent_fault_90d_counts = []
recent_disturbance_30d_counts = []

for row in contract_df.itertuples(index=False):
        ref_time = row.timestamp if pd.notna(row.timestamp) else row.event_end
        fault_rows = fault_alignment[
            fault_alignment['manufacturer'].eq(row.manufacturer)
            & fault_alignment['substation_id'].eq(row.substation_id)
            & fault_alignment['report_date'].notna()
            & fault_alignment['report_date'].lt(ref_time)
        ].sort_values('report_date', ascending=False)
        disturbance_rows = disturbance_alignment[
            disturbance_alignment['manufacturer'].eq(row.manufacturer)
            & disturbance_alignment['substation_id'].eq(row.substation_id)
            & disturbance_alignment['event_start'].notna()
            & disturbance_alignment['event_start'].lt(ref_time)
        ].sort_values('event_start', ascending=False)

        recent_fault_90d = fault_rows[fault_rows['report_date'].ge(ref_time - pd.Timedelta(days=90))]
        recent_disturbance_30d = disturbance_rows[disturbance_rows['event_start'].ge(ref_time - pd.Timedelta(days=30))]

        recent_fault_90d_counts.append(int(len(recent_fault_90d)))
        recent_disturbance_30d_counts.append(int(len(recent_disturbance_30d)))

        fault_payload = {
            'previous_fault_count': int(len(fault_rows)),
            'recent_fault_90d_count': int(len(recent_fault_90d)),
            'latest_fault_label': None if fault_rows.empty else fault_rows.iloc[0]['fault_label'],
            'latest_fault_report_date': None if fault_rows.empty else fault_rows.iloc[0]['report_date'].isoformat(),
        }
        disturbance_payload = {
            'previous_disturbance_count': int(len(disturbance_rows)),
            'recent_disturbance_30d_count': int(len(recent_disturbance_30d)),
            'latest_disturbance_type': None if disturbance_rows.empty else disturbance_rows.iloc[0]['disturbance_type'],
            'latest_disturbance_start': None if disturbance_rows.empty else disturbance_rows.iloc[0]['event_start'].isoformat(),
        }
        related_fault_history.append(json.dumps(fault_payload, ensure_ascii=False))
        related_disturbance_history.append(json.dumps(disturbance_payload, ensure_ascii=False))

contract_df['related_fault_history'] = related_fault_history
contract_df['related_disturbance_history'] = related_disturbance_history
contract_df['recent_fault_90d_count'] = recent_fault_90d_counts
contract_df['recent_disturbance_30d_count'] = recent_disturbance_30d_counts

contract_df['normalized_max_anomaly_score'] = contract_df['max_anomaly_score'].fillna(0.0).clip(lower=0.0, upper=4.0).map(lambda value: np.log1p(value) / np.log1p(4.0))
contract_df['point_anomaly_density'] = (
    contract_df['p99_above_count'].fillna(0.0)
    / contract_df['selected_window_count'].replace(0, np.nan)
).fillna(0.0).clip(lower=0.0, upper=1.0)
contract_df['criticality_score'] = contract_df['max_counter'].fillna(0.0).clip(lower=0.0, upper=2.0) / 2.0
contract_df['detection_signal'] = contract_df['detected'].astype(float)
contract_df['history_signal'] = (
    0.6 * contract_df['recent_fault_90d_count'].clip(upper=2) / 2.0
    + 0.4 * contract_df['recent_disturbance_30d_count'].clip(upper=3) / 3.0
)
contract_df['base_risk_score'] = (
    0.25 * contract_df['normalized_max_anomaly_score']
    + 0.20 * contract_df['point_anomaly_density']
    + 0.20 * contract_df['criticality_score']
    + 0.35 * contract_df['detection_signal']
).clip(upper=1.0)
contract_df['risk_score'] = (
    contract_df['base_risk_score']
    * (0.45 + 0.55 * contract_df['detection_signal'])
).clip(upper=1.0)
contract_df['risk_level'] = contract_df['risk_score'].map(risk_level_from_score)
contract_df['priority_score'] = (
    100.0
    * (
        0.75 * contract_df['risk_score']
        + 0.25 * contract_df['history_signal']
    ).clip(upper=1.0)
).round(2)

contract_df = contract_df.sort_values(['priority_score', 'risk_score', 'timestamp'], ascending=[False, False, False]).reset_index(drop=True)
contract_df['priority_rank'] = np.arange(1, len(contract_df) + 1)
contract_df['priority_reason'] = contract_df.apply(
    lambda row: '; '.join(
        [
            f"max_anomaly_score={row.max_anomaly_score:.3f}",
            f"criticality_score={row.criticality_score:.3f}",
            f"detected={bool(row.detected)}",
            f"recent_fault_90d={int(row.recent_fault_90d_count)}",
            f"recent_disturbance_30d={int(row.recent_disturbance_30d_count)}",
        ]
    ),
    axis=1,
)

contract_df['anomaly_score'] = contract_df['max_anomaly_score'].fillna(0.0)
contract_df['is_detected'] = contract_df['detected'].astype(bool)

agent_columns = [
    'substation_id',
    'timestamp',
    'event_id',
    'event_type',
    'manufacturer',
    'configuration_type',
    'anomaly_score',
    'risk_score',
    'risk_level',
    'criticality_score',
    'is_detected',
    'main_abnormal_features',
    'related_fault_history',
    'related_disturbance_history',
    'feature_explanation',
    'priority_score',
    'priority_rank',
    'priority_reason',
    'lead_time_hours',
    'fault_label',
    'event_split',
]
agent_output_df = contract_df[agent_columns].copy()
agent_output_df.to_csv(AGENT_OUTPUT, index=False, encoding='utf-8-sig')

priority_output_df = contract_df[[
    'manufacturer',
    'substation_id',
    'timestamp',
    'event_id',
    'event_type',
    'anomaly_score',
    'criticality_score',
    'risk_score',
    'risk_level',
    'priority_score',
    'priority_rank',
    'recent_fault_90d_count',
    'recent_disturbance_30d_count',
    'is_detected',
    'priority_reason',
]].copy()
priority_output_df.to_csv(PRIORITY_OUTPUT, index=False, encoding='utf-8-sig')

schema = {
    'version': 'paper_aligned_agent_contract_v1',
    'primary_key': ['manufacturer', 'substation_id', 'event_id', 'event_type'],
    'fields': agent_columns,
    'risk_score_definition': '(0.25 * normalized_max_anomaly_score + 0.20 * point_anomaly_density + 0.20 * criticality_score + 0.35 * detection_signal) * (0.45 + 0.55 * detection_signal)',
    'priority_score_definition': '100 * (0.75 * risk_score + 0.25 * history_signal)',
    'risk_level_thresholds': {
        'low': '< 0.30',
        'medium': '0.30 - 0.49',
        'high': '0.50 - 0.74',
        'critical': '>= 0.75',
    },
}
SCHEMA_OUTPUT.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

metadata = {
    'contract_version': 'paper_aligned_agent_contract_v1',
    'history_windows': {
        'fault_days': 90,
        'disturbance_days': 30,
    },
    'note': 'risk_score is an operator-facing prioritisation signal, not a fault probability',
}
METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

display(agent_output_df.head(10))
display(priority_output_df.head(10))
display(contract_df['risk_level'].value_counts().rename_axis('risk_level').reset_index(name='count'))
print(f'saved: {AGENT_OUTPUT}')
print(f'saved: {PRIORITY_OUTPUT}')
print(f'saved: {SCHEMA_OUTPUT}')
print(f'saved: {METADATA_OUTPUT}')


,substation_id,timestamp,event_id,event_type,manufacturer,configuration_type,anomaly_score,risk_score,risk_level,criticality_score,is_detected,main_abnormal_features,related_fault_history,related_disturbance_history,feature_explanation,priority_score,priority_rank,priority_reason,lead_time_hours,fault_label,event_split
0,30,2020-03-13 00:00:00,10,fault_event,manufacturer 1,SH + DHW,5.874937,1.000000,critical,1.000000,True,dhw supply temperature gap / mean (lower_than_...,"{""previous_fault_count"": 0, ""recent_fault_90d_...","{""previous_disturbance_count"": 2, ""recent_dist...",dhw supply temperature gap / mean (lower_than_...,75.00,1,max_anomaly_score=5.875; criticality_score=1.0...,87.066667,Leakage,validation
1,10,2014-05-04 00:00:00,1,fault_event,manufacturer 1,SH + DHW,5.184350,1.000000,critical,1.000000,True,dhw supply temperature gap / mean (lower_than_...,"{""previous_fault_count"": 0, ""recent_fault_90d_...","{""previous_disturbance_count"": 2, ""recent_dist...",dhw supply temperature gap / mean (lower_than_...,75.00,2,max_anomaly_score=5.184; criticality_score=1.0...,14.733333,Motorised control valve (primary side): Actuat...,holdout
2,31,2020-01-11 00:00:00,45,fault_event,manufacturer 1,SH + DHW,17.246949,0.963636,critical,1.000000,True,s dhw supply temperature setpoint / std (highe...,"{""previous_fault_count"": 0, ""recent_fault_90d_...","{""previous_disturbance_count"": 0, ""recent_dist...",s dhw supply temperature setpoint / std (highe...,72.27,3,max_anomaly_score=17.247; criticality_score=1....,63.800000,"Heat exchanger: Leakage, external",validation
3,26,2020-06-13 00:00:00,7,fault_event,manufacturer 1,SH + DHW with sub-circuits,2.897641,0.961312,critical,1.000000,True,dhw supply temperature gap / mean (lower_than_...,"{""previous_fault_count"": 1, ""recent_fault_90d_...","{""previous_disturbance_count"": 4, ""recent_dist...",dhw supply temperature gap / mean (lower_than_...,72.10,4,max_anomaly_score=2.898; criticality_score=1.0...,10.633333,Incorrect setting of the differential pressure...,validation
4,18,2020-02-01 12:00:00,7,fault_event,manufacturer 2,SH with buffer tank,2.476939,0.707930,high,0.738470,True,s dhw lower storage temperature / min (higher_...,"{""previous_fault_count"": 1, ""recent_fault_90d_...","{""previous_disturbance_count"": 2, ""recent_dist...",s dhw lower storage temperature / min (higher_...,67.26,5,max_anomaly_score=2.477; criticality_score=0.7...,68.550000,Leakage,validation
5,13,2020-05-25 06:00:00,53,fault_event,manufacturer 1,SH + DHW,2.811502,0.778990,critical,0.905751,True,s dhw supply temperature setpoint / last (high...,"{""previous_fault_count"": 1, ""recent_fault_90d_...","{""previous_disturbance_count"": 4, ""recent_dist...",s dhw supply temperature setpoint / last (high...,58.42,6,max_anomaly_score=2.812; criticality_score=0.9...,28.000000,Shut-off valve closed,holdout
6,59,2020-01-21 18:00:00,8,fault_event,manufacturer 2,SH,2.561630,0.720137,high,0.780815,True,p hc1 return temperature / std (higher_than_no...,"{""previous_fault_count"": 0, ""recent_fault_90d_...","{""previous_disturbance_count"": 1, ""recent_dist...",p hc1 return temperature / std (higher_than_no...,54.01,7,max_anomaly_score=2.562; criticality_score=0.7...,68.766667,Failure of the heating circuit pump,validation
7,11,2018-11-20 12:00:00,5,fault_event,manufacturer 1,SH + DHW,2.271797,0.677970,high,0.635899,True,s dhw lower storage temperature / min (higher_...,"{""previous_fault_count"": 1, ""recent_fault_90d_...","{""previous_disturbance_count"": 4, ""recent_dist...",s dhw lower storage temperature / min (higher_...,50.85,8,max_anomaly_score=2.272; criticality_score=0.6...,68.500000,Failure of the heating circuit pump,holdout
8,26,2020-03-10 00:00:00,69,fault_event,manufacturer 1,SH + DHW with sub-circuits,1.692826,0.647354,high,0.535592,True,dhw supply temperature gap / mean (lower_than_...,"{""previous_fault_count"": 0, ""recent_fault_90d_...","{""previous_disturbance_count"": 2, ""recent_dist...",dhw 

,manufacturer,substation_id,timestamp,event_id,event_type,anomaly_score,criticality_score,risk_score,risk_level,priority_score,priority_rank,recent_fault_90d_count,recent_disturbance_30d_count,is_detected,priority_reason
0,manufacturer 1,30,2020-03-13 00:00:00,10,fault_event,5.874937,1.000000,1.000000,critical,75.00,1,0,0,True,max_anomaly_score=5.875; criticality_score=1.0...
1,manufacturer 1,10,2014-05-04 00:00:00,1,fault_event,5.184350,1.000000,1.000000,critical,75.00,2,0,0,True,max_anomaly_score=5.184; criticality_score=1.0...
2,manufacturer 1,31,2020-01-11 00:00:00,45,fault_event,17.246949,1.000000,0.963636,critical,72.27,3,0,0,True,max_anomaly_score=17.247; criticality_score=1....
3,manufacturer 1,26,2020-06-13 00:00:00,7,fault_event,2.897641,1.000000,0.961312,critical,72.10,4,0,0,True,max_anomaly_score=2.898; criticality_score=1.0...
4,manufacturer 2,18,2020-02-01 12:00:00,7,fault_event,2.476939,0.738470,0.707930,high,67.26,5,1,2,True,max_anomaly_score=2.477; criticality_score=0.7...
5,manufacturer 1,13,2020-05-25 06:00:00,53,fault_event,2.811502,0.905751,0.778990,critical,58.42,6,0,0,True,max_anomaly_score=2.812; criticality_score=0.9...
6,manufacturer 2,59,2020-01-21 18:00:00,8,fault_event,2.561630,0.780815,0.720137,high,54.01,7,0,0,True,max_anomaly_score=2.562; criticality_score=0.7...
7,manufacturer 1,11,2018-11-20 12:00:00,5,fault_event,2.271797,0.635899,0.677970,high,50.85,8,0,0,True,max_anomaly_score=2.272; criticality_score=0.6...
8,manufacturer 1,26,2020-03-10 00:00:00,69,fault_event,1.692826,0.535592,0.647354,high,48.55,9,0,0,True,max_anomaly_score=1.693; criticality_score=0.5...
9,manufacturer 2,53,2020-03-23 15:19:00,0,fault_event,0.526017,0.000000,0.029544,low,16.38,10,1,2,False,max_anomaly_score=0.526; criticality_score=0.0...


,risk_level,count
0,low,35
1,critical,5
2,high,4


saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\agent_contract_output.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\priority_engine_input.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\agent_contract_schema.json
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\agent_contract_metadata.json


## 종료 상태

- 06-P 체인의 canonical contract output이 생성됐다.
- 이후 `07`, `08` 단계에서는 이 계약을 설명/내보내기 계층으로 연결하면 된다.
